# Summarization Memory

Summarization memory is a technique for managing long conversations by
compressing older conversation history into a shorter summary while preserving
the important context.

In normal conversation memory, the complete history of `HumanMessage` and
`AIMessage` is stored and sent to the LLM for every new message. As the
conversation grows, the history becomes larger, increasing the context size
and the amount of information the LLM needs to process.

```text
HumanMessage
      ↓
AIMessage
      ↓
HumanMessage
      ↓
AIMessage
      ↓
...
```


In [12]:
# Previous notebook ko code reuse garna
%run ./memory/17_conversation_memory.ipynb

LLM loaded successfully!
[]
[HumanMessage(content='My name is Arjun.', additional_kwargs={}, response_metadata={})]
Nice to meet you, Arjun!
[HumanMessage(content='My name is Arjun.', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Arjun!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Your name is Arjun!
HumanMessage: My name is Arjun.
AIMessage: Nice to meet you, Arjun!
HumanMessage: What is my name?
AIMessage: Your name is Arjun!
Assistant: Python is a fantastic language! Many people share your enthusiasm for Python, and it's easy to see why. Python is a high-level language that's known for its simplicity, readability, and ease of use. It's a great language for beginners and experienced programmers alike.

What is it about Python that you enjoy the most? Is it the syntax, the vast number of libraries and frameworks, or something else entirely?

Also, what kind of projects or areas of programming do you enjoy work

In [13]:
# Older conversation lai summarize garna prompt
summary_prompt = """
You are a conversation memory summarizer.

Your task is to summarize ONLY the information explicitly stated in the
conversation.

IMPORTANT RULES:
- Treat the conversation as the only source of truth.
- Do not use outside knowledge.
- Do not explain concepts mentioned by the user.
- Do not add definitions or facts that the user did not state.
- Do not infer additional information.
- Do not correct the user.
- Preserve the user's exact meaning.
- Keep important user facts, preferences, goals, requirements, and decisions.
- Remove small talk and repetitive information.

Conversation:
{conversation}

Write a short factual summary containing only information explicitly present
in the conversation.

Summary:
"""

In [14]:
# Conversation messages lai readable text ma convert garne
def format_messages(messages):

    formatted = []

    for message in messages:

        # Message ko role identify garne
        if isinstance(message, HumanMessage):
            role = "User"

        elif isinstance(message, AIMessage):
            role = "Assistant"

        else:
            role = "Message"

        formatted.append(f"{role}: {message.content}")

    return "\n".join(formatted)

In [15]:
# Conversation messages lai summarize garne
def summarize_messages(messages):

    # Messages lai text format ma convert garne
    conversation = format_messages(messages)

    # Summarization prompt prepare garne
    prompt = summary_prompt.format(conversation=conversation)

    # LLM bata summary generate garne
    response = llm.invoke(prompt)

    return response.content

In [16]:
# Summarization memory manage garne class
class SummarizationMemory:

    def __init__(self, max_recent_messages=4):

        # Older conversation ko summary store garne
        self.summary = ""

        # Recent messages store garne
        self.recent_messages = []

        # Recent messages ko maximum limit
        self.max_recent_messages = max_recent_messages

    def add_message(self, message):

        # New message recent messages ma add garne
        self.recent_messages.append(message)

        # Limit exceed bhayo bhane older messages summarize garne
        if len(self.recent_messages) > self.max_recent_messages:

            self._summarize_old_messages()

    def _summarize_old_messages(self):

        # Recent messages bhanda agadi ko messages select garne
        old_messages = self.recent_messages[: -self.max_recent_messages]

        # Existing summary cha bhane teslai pani include garne
        if self.summary:

            old_messages = [
                HumanMessage(content=f"Existing summary:\n{self.summary}")
            ] + old_messages

        # Older conversation ko updated summary generate garne
        self.summary = summarize_messages(old_messages)

        # Recent messages matra retain garne
        self.recent_messages = self.recent_messages[-self.max_recent_messages :]

    def get_context(self):

        # LLM lai dine context prepare garne
        context = []

        # Summary available cha bhane context ma add garne
        if self.summary:

            context.append(
                HumanMessage(content=f"Conversation summary:\n{self.summary}")
            )

        # Recent messages context ma add garne
        context.extend(self.recent_messages)

        return context

In [17]:
# Summarization memory use garera conversation handle garne
def chat_with_summarization_memory(user_message, memory):

    # User ko message create garne
    human_message = HumanMessage(content=user_message)

    # User message memory ma add garne
    memory.add_message(human_message)

    # Summary ra recent messages retrieve garne
    context = memory.get_context()

    # Compressed conversation context LLM lai diyera response generate garne
    response = llm.invoke(context)

    # Assistant ko response memory ma store garne
    memory.add_message(AIMessage(content=response.content))

    # Generated response return garne
    return response.content

In [18]:
# Summarization memory create garne
memory = SummarizationMemory(max_recent_messages=4)

In [19]:
# Summarization memory reset garne
memory = SummarizationMemory(max_recent_messages=4)

# Long conversation simulate garne
messages = [
    "My name is Arjun.",
    "I am a IT engineering student.",
    "I am learning Python and machine learning.",
    "I am currently studying RAG systems.",
    "I want to build a fully local RAG chatbot.",
    "I am learning different RAG techniques before combining them.",
]

# Conversation lai summarization memory sanga process garne
for message in messages:
    response = chat_with_summarization_memory(message, memory)

    print(f"\nUser: {message}")
    print(f"Assistant: {response}")


User: My name is Arjun.
Assistant: Nice to meet you, Arjun! What brings you here today?

User: I am a IT engineering student.
Assistant: That's great! As an IT engineering student, you're likely to be interested in the latest technologies and innovations in the field. What are some of the areas of IT that you're most interested in or passionate about? Is it software development, data science, cybersecurity, or something else?

User: I am learning Python and machine learning.
Assistant: Python is a fantastic language to learn, and machine learning is a fascinating field with many applications. Python is a popular choice for machine learning due to its simplicity, flexibility, and extensive libraries like scikit-learn and TensorFlow.

What drew you to Python and machine learning? Are you working on any projects or exploring specific areas like computer vision, natural language processing, or recommender systems?

User: I am currently studying RAG systems.
Assistant: RAG systems! That's 

In [20]:
# Check whether older conversation has been summarized
print("SUMMARY:")
print(memory.summary)

print("\nRECENT MESSAGES:")

for message in memory.recent_messages:
    print(f"{message.__class__.__name__}: " f"{message.content}")

SUMMARY:
Here is the short factual summary:

Arjun's name is Arjun.
The user is learning Python and machine learning.
The user is currently studying RAG systems.

RECENT MESSAGES:
HumanMessage: I want to build a fully local RAG chatbot.
AIMessage: A fully local RAG chatbot sounds like an exciting project! Building a chatbot that can operate independently without relying on cloud-based services or external APIs requires a deep understanding of natural language processing (NLP), machine learning, and computer vision.

To create a fully local RAG chatbot, you'll need to develop a system that can process and understand user input, generate responses, and make decisions without relying on external data or services. This might involve using techniques like:

1. Intent recognition: Identifying the user's intent or goal from their input.
2. Entity recognition: Identifying specific entities like names, locations, or objects mentioned in the user's input.
3. Dialogue management: Managing the con

In [21]:
# Older conversation ko information recall garna question sodheko
response = chat_with_summarization_memory(
    "What am I planning to build and what technologies do I want to use?", memory
)

print("\nAssistant:", response)


Assistant: You're planning to build a fully local RAG (Robotics, Artificial Intelligence, and Guidance) chatbot, and you want to use specific technologies to bring your project to life.

From our previous conversation, I understand that you're focusing on learning different RAG techniques before combining them to build your chatbot. You're likely exploring various aspects of autonomous guidance, such as sensor fusion, motion planning, decision-making, and object recognition.

To build your fully local RAG chatbot, you might be considering using technologies like:

1. Python: A popular programming language for AI and robotics projects, with libraries like OpenCV, NumPy, and scikit-learn.
2. R: A programming language and environment for statistical computing and graphics, which can be used for data analysis and visualization.
3. TensorFlow or PyTorch: Deep learning frameworks for building and training AI models.
4. OpenCV: A computer vision library for image and video processing, object